# Hypothesis Tests
 
This notebook finalises the pre-registered confirmatory analyses H1–H4 - detailed below.   
**OSF project link** : https://osf.io/fkb87      
**OSF pre-registration link** : https://osf.io/uj92x/overview

**Preregistered Hypotheses**:
1. SC-R discrimination - Self-causal (SC) property press rate will significantly exceed random (R) property press rate in block 2, across both tasks. This tests whether participants learn to distinguish self-caused from random outcomes within 60 trials.

2. Transfer learning - Task 2 block 2 SC press rate in the first 15 SC-present trials will exceed Task 1 block 2 SC press rate in the first 15 SC-present trials (transfer index > 0). This tests whether the property-indexed causal efficacy belief formed in Task 1 carries forward to Task 2 despite changed task context and colour values.

3. Observation priming - Group A participants (who observe OC causal structure during block 1 watch trials) will reach SC-R press rate criterion faster in block 2 than Group B participants (who observe random outcomes). This tests whether observing environmental causal structure seeds self-efficacy belief formation.

4. Belief persistence - Block 3 SC press rate will significantly exceed block 1 baseline press rate, indicating that property-indexed causal efficacy beliefs persist in the absence of outcome feedback.

*Secondary H test*: SC property type (colour, size, velocity) will be tested as a moderator of SC-R discrimination magnitude, based on pilot evidence (N=5, prior to full data collection) suggesting velocity produces weaker learning signal than colour or size.

**Multiple comparisons**: 4 primary tests → Bonferroni α = 0.05 / 4 = **0.0125**.  
All primary tests are reported at both uncorrected (α = 0.05) and corrected (α = 0.0125) thresholds.  
Effect sizes selected : Cohen's *d* - all hypotheses were tested using *t*-tests. 

------

### Summary

| Test | IV | DV | Design | Statistical Test | Effect Size | Tail | Power Target |
|------|---------|---------|-----------|---------|------|------|------|
| H1 | Condition-dependent response type | Press rate | Within-subject + paired comparison | Paired samples t-test | dz | One (SC > R) | 0.80 |
| H2 | N.A. (sample mean vs. reference value) | Transfer index for SC press rate | Within-subject | One-sample t-test | *d* | One (> 0) | 0.80 |
| H3 | Group assignment | SC acquisition rate (slope) | Between-subjects | Independent-samples t-test | Independent-samples Cohen *d* | Two | 0.80 | 
| H4 | Block | SC press rate (B2 → B3 change) | Within-subject + repeated measure | Paired samples t-test | Paired-samples Cohen's *d* | Two | 0.80 |

**† H3 note**: The preregistered "trials to SC-R criterion" measure could not be reliably computed due to R trial sparsity (~10% of block 2). SC acquisition slope was used as an alternative, supported by the preregistration's Indices section.  
**† H4 note**: Block 1 press rate = 0 by design (watch-only, spacebar disabled). The meaningful persistence test compares block 3 SC press rate to block 2 (both mean and late-block values). Both comparisons are reported.

In [23]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import linregress, ttest_ind, ttest_1samp, ttest_rel
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [24]:
alpha = 0.05
alpha_bonferroni = 0.0125
min_trials = 5


# Same functions, data structures, and objects as initially set up in 02_EDA.ipynb analysis - see notebook for initial script

df_clean = pd.read_csv('../data/processed/clean_data.csv')
df_clean = df_clean.rename(columns={'prop_SC': 'sc_prop', 'prop_OC': 'oc_prop', 'prop_R': 'r_prop'})

trials = df_clean[df_clean['watch_only'] == 0].copy()
trials['pressed'] = trials['pressed'].astype(int)
trials['outcome'] = trials['outcome'].astype(int)

_t1   = df_clean[df_clean['task'] == 1].groupby('subject_id').first().reset_index()
_pall = trials.groupby('subject_id')['pressed'].mean().rename('overall_press_rate')
subject_meta = (
    _t1[['subject_id', 'group', 'sc_prop', 'oc_prop', 'r_prop', 'completion_min']]
    .merge(_pall, on='subject_id')
)

press_rates = (
    trials
    .groupby(['subject_id', 'group', 'sc_prop', 'task', 'block', 'trial_role'])
    .agg(
        n_trials=('pressed', 'count'),
        n_pressed=('pressed', 'sum'),
        press_rate=('pressed', 'mean'),
        mean_rt=('rt', 'mean'),
    )
    .reset_index()
)


# Analysis-specific press-rate table for H1
# Other hypotheses below use trial-level data directly, so they do not inherit this min. trial filter

pr_b2_min5 = press_rates[
    (press_rates['block'] == 2) &
    (press_rates['n_trials'] >= min_trials)
].copy()


def one_tailed_p(t_stat, p_two, direction='positive'):
    if direction == 'positive':
        return p_two / 2 if t_stat > 0 else 1 - p_two / 2
    else:
        return p_two / 2 if t_stat < 0 else 1 - p_two / 2

def cohens_d_paired(diff_series):
    return diff_series.mean() / diff_series.std(ddof=1)

def cohens_d_ind(a, b):
    n_a, n_b = len(a), len(b)
    pooled = np.sqrt(
        ((n_a - 1) * a.std(ddof=1)**2 + (n_b - 1) * b.std(ddof=1)**2) / (n_a + n_b - 2)
    )
    return (a.mean() - b.mean()) / pooled


print(f'Subjects: {trials["subject_id"].nunique()}')
print(f'Active trials: {len(trials):,}')
print(f'Block 2 PR cells (min_trials >= {min_trials}): {len(pr_b2_min5)} / {len(press_rates[press_rates["block"] == 2])}')
print()
print('Group × SC property (N imbalance — note for H3):')
print(pd.crosstab(subject_meta['group'], subject_meta['sc_prop'], margins=True))


Subjects: 70
Active trials: 11,200
Block 2 PR cells (min_trials >= 5): 404 / 420

Group × SC property (N imbalance — note for H3):
sc_prop  colour  size  velocity  All
group                               
A            12     9        18   39
B            10    16         5   31
All          22    25        23   70


## H1: SC–R discrimination

- Preregistration : SC press rate > R press rate in block 2, indiscriminate of tasks.  
- Statistic : paired t-test on per-subject means.  
- Tails : one-tailed (SC > R).  
- Rationale : tests that participants have learned to discriminate the SC contingency from the random baseline within the 60 active trials of block 2.

SC and R press rates are averaged across tasks per subject before the paired test.  
The per-subject mean is computed from `pr_b2_min5`, which applies `min_trials ≥ 5` only to Block 2 SC/R press-rate cells used for H1. Other hypotheses use trial-level data directly where appropriate.


In [25]:
# Per-subject, per-task SC and R press rates in block 2

h1_by_task = (
    pr_b2_min5[pr_b2_min5['trial_role'].isin(['SC', 'R'])]
    .pivot_table(index=['subject_id', 'task'], columns='trial_role', values='press_rate')
    .reset_index()
    .dropna(subset=['SC', 'R'])
)

h1_subj = (
    h1_by_task.groupby('subject_id')[['SC', 'R']].mean().reset_index()
)

diff_h1 = h1_subj['SC'] - h1_subj['R']
t_h1, p2_h1 = stats.ttest_rel(h1_subj['SC'], h1_subj['R'])
p1_h1 = one_tailed_p(t_h1, p2_h1, 'positive')
d_h1 = cohens_d_paired(diff_h1)
ci_lo, ci_hi = stats.t.interval(0.95, df=len(diff_h1) - 1,
                                     loc=diff_h1.mean(), scale=stats.sem(diff_h1))

print(f'SC press rate: M = {h1_subj["SC"].mean():.3f}, SD = {h1_subj["SC"].std():.3f}')
print(f'R  press rate: M = {h1_subj["R"].mean():.3f}, SD = {h1_subj["R"].std():.3f}')
print()
print(f'Mean diff (SC-R): M = {diff_h1.mean():.3f}, SD = {diff_h1.std():.3f}')
print(f'95% CI on diff: [{ci_lo:.3f}, {ci_hi:.3f}]')
print()
print(f't({len(h1_subj)-1}) = {t_h1:.3f}')
print(f'p (two-tailed) = {p2_h1:.4f}')
print(f'p (one-tailed) = {p1_h1:.4f}')
print(f"Cohen's d = {d_h1:.3f}")
print()
print(f'Significant at alpha = {alpha} (one-tailed): {p1_h1 < alpha}')
print(f'Significant at Bonferroni alpha = {alpha_bonferroni}: {p1_h1 < alpha_bonferroni}')
print()
print()


# Task-stratified breakdown (descriptive, not pre-registered)

print('Task-stratified SC - R press rates (descriptive)')
for task in [1, 2]:
    d = h1_by_task[h1_by_task['task'] == task].dropna(subset=['SC', 'R'])
    diff_t = d['SC'] - d['R']
    t_t, p_t = stats.ttest_rel(d['SC'], d['R'])
    p1_t = one_tailed_p(t_t, p_t, 'positive')
    print(f'Task {task} (N={len(d)}):  SC={d["SC"].mean():.3f}, R={d["R"].mean():.3f},'
          f'diff={diff_t.mean():.3f}, t={t_t:.3f}, p(one-tail)={p1_t:.4f}')
print()

# Figures

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
for _, row in h1_subj.iterrows():
    ax.plot([0, 1], [row['SC'], row['R']], color='steelblue', alpha=0.18, linewidth=0.9)
ax.plot([0, 1], [h1_subj['SC'].mean(), h1_subj['R'].mean()],
        color='steelblue', linewidth=2.5, marker='o', markersize=8, label='group mean')
ax.set(xticks=[0, 1], xticklabels=['SC', 'R'],
       ylabel='Press rate (block 2)',
       title='H1: SC vs R — paired means (collapsed across tasks)')
ax.legend()

ax = axes[1]
ax.hist(diff_h1, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='no difference')
ax.axvline(diff_h1.mean(), color='tomato', linewidth=2,
           label=f'mean diff = {diff_h1.mean():.3f}')
ax.set(xlabel='SC press rate − R press rate', ylabel='Subjects',
       title='H1: distribution of SC−R differences')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../output/h1_sc_r_paired.png', dpi=150, bbox_inches='tight')
print("Figure saved: 'output/h1_sc_r_paired.png'")


SC press rate: M = 0.469, SD = 0.236
R  press rate: M = 0.404, SD = 0.211

Mean diff (SC-R): M = 0.065, SD = 0.242
95% CI on diff: [0.007, 0.122]

t(69) = 2.239
p (two-tailed) = 0.0284
p (one-tailed) = 0.0142
Cohen's d = 0.268

Significant at alpha = 0.05 (one-tailed): True
Significant at Bonferroni alpha = 0.0125: False


Task-stratified SC - R press rates (descriptive)
Task 1 (N=69):  SC=0.501, R=0.416,diff=0.085, t=2.295, p(one-tail)=0.0124
Task 2 (N=66):  SC=0.429, R=0.380,diff=0.049, t=1.457, p(one-tail)=0.0749

Figure saved: 'output/h1_sc_r_paired.png'


Across tasks within block 2, SC press rates were higher than R press rates, with a mean difference of 0.065. This difference was statistically significant under the pre-registered one-tailed alpha = 0.05 threshold (t(69) = 2.239, p = 0.0142 one-tailed), but did not survive
Bonferroni correction (alpha = 0.0125).

The estimated effect size was small, at d = 0.268, and the 95% CI on the mean difference was [0.007 - 0.122]

A post-hoc analysis restricted to Task 1 yielded p=0.0124, d=0.276, which marginally survives correction. 
The exploratory analysis was motivated by the observation that OC/R role reassignment in Task 2 introduces additional variance in R press rates. Task 1 results are reported for completeness but were not pre-specified.

## H2: Transfer index

- Preregistration : transfer index > 0 (Task 2 early SC press rate exceeds Task 1 early SC press rate).  
- Transfer index : Task 2 block 2 SC press rate (first 15 SC-present trials) minus Task 1 block 2 SC press rate (first 15 SC-present trials), per subject.  
- Statistic : one-sample t-test vs 0, one-tailed (> 0).  
- Rationale : if property-indexed causal beliefs transfer from Task 1 to Task 2, participants should press more on SC trials at the start of Task 2 block 2 than they did at the start of Task 1 block 2. The first-15-SC-trials window captures this prior to new Task 2 learning.  

All 70 subjects have ≥33 SC trials in block 2 per task, so no exclusions apply here.

*Secondary*: logistic GEE model on SC-present block 2 trials, characterising whether learning curve shape differs between tasks.

In [26]:
# First 15 SC-present trials in block 2 x subject × task

sc_b2_first15 = (
    trials[(trials['block'] == 2) & (trials['trial_role'] == 'SC')]
    .sort_values(['subject_id', 'task', 'trial_n'])
    .groupby(['subject_id', 'task'], group_keys=False)
    .head(15)
)

sc_early = (
    sc_b2_first15
    .groupby(['subject_id', 'task'])
    .agg(n_sc_trials=('pressed', 'count'), press_rate=('pressed', 'mean'))
    .reset_index()
)
print(f'SC trials used per group: min={sc_early["n_sc_trials"].min()}, max={sc_early["n_sc_trials"].max()}')

ti_wide = (
    sc_early
    .pivot_table(index='subject_id', columns='task', values='press_rate')
    .rename(columns={1: 'task1', 2: 'task2'})
    .reset_index()
    .dropna()
)
ti_wide['transfer_idx'] = ti_wide['task2'] - ti_wide['task1']
ti = ti_wide['transfer_idx']

t_h2, p2_h2 = stats.ttest_1samp(ti, 0)
p1_h2 = one_tailed_p(t_h2, p2_h2, 'positive')
d_h2 = cohens_d_paired(ti)
ci_lo2, ci_hi2 = stats.t.interval(0.95, df=len(ti) - 1,
                                    loc=ti.mean(), scale=stats.sem(ti))

print()
print(f'Task 1 early SC rate: M = {ti_wide["task1"].mean():.3f}, SD = {ti_wide["task1"].std():.3f}')
print(f'Task 2 early SC rate: M = {ti_wide["task2"].mean():.3f}, SD = {ti_wide["task2"].std():.3f}')
print()
print(f'Transfer index: M = {ti.mean():.3f}, Median = {ti.median():.3f}, SD = {ti.std():.3f}')
print(f'95% CI: [{ci_lo2:.3f}, {ci_hi2:.3f}]')
print(f'% positive: {(ti > 0).mean()*100:.1f}%   % negative: {(ti < 0).mean()*100:.1f}%')
print()
print(f't({len(ti)-1}) = {t_h2:.3f}')
print(f'p (two-tailed) = {p2_h2:.4f}')
print(f'p (one-tailed) = {p1_h2:.4f}')
print(f"Cohen's d = {d_h2:.3f}")
print()
print(f'Significant at alpha = {alpha} (one-tailed): {p1_h2 < alpha}')
print(f'Significant at Bonferroni alpha = {alpha_bonferroni}: {p1_h2 < alpha_bonferroni}')
print()

ti_wide = ti_wide.merge(subject_meta[['subject_id', 'sc_prop', 'group']], on='subject_id')


# Figures

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
prop_colours = {'colour': 'mediumorchid', 'size': 'steelblue', 'velocity': 'seagreen'}
for prop, col in prop_colours.items():
    d = ti_wide[ti_wide['sc_prop'] == prop]
    mk = {'colour': 'o', 'size': 's', 'velocity': '^'}[prop]
    ax.scatter(d['task1'], d['task2'], color=col, marker=mk, alpha=0.65,
               label=f'{prop} (n={len(d)})', s=50, zorder=3)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='no change')
ax.set(xlabel='Task 1 SC press rate (first 15 trials)',
       ylabel='Task 2 SC press rate (first 15 trials)',
       title='H2: SC early press rate — Task 1 vs Task 2',
       xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))
ax.legend(fontsize=8)
ax = axes[1]
ax.hist(ti, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='TI = 0 (no transfer)')
ax.axvline(ti.mean(), color='tomato', linewidth=2,
           label=f'mean = {ti.mean():.3f}')
ax.set(xlabel='Transfer index (Task 2 − Task 1 early SC rate)',
       ylabel='Subjects', title='H2: transfer index distribution')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../output/h2_transfer_index.png', dpi=150, bbox_inches='tight')
print("Figure saved: 'output/h2_transfer_index.png'")

SC trials used per group: min=15, max=15

Task 1 early SC rate: M = 0.465, SD = 0.273
Task 2 early SC rate: M = 0.412, SD = 0.304

Transfer index: M = -0.052, Median = -0.033, SD = 0.306
95% CI: [-0.125, 0.021]
% positive: 35.7%   % negative: 50.0%

t(69) = -1.432
p (two-tailed) = 0.1566
p (one-tailed) = 0.9217
Cohen's d = -0.171

Significant at alpha = 0.05 (one-tailed): False
Significant at Bonferroni alpha = 0.0125: False

Figure saved: 'output/h2_transfer_index.png'


In [27]:
# H2 Secondary: Logistic GEE — pressed ~ trial_c * task_cat

sc_b2_gee = trials[
    (trials['block'] == 2) &
    (trials['trial_role'] == 'SC')
].copy()

sc_b2_gee['pressed'] = sc_b2_gee['pressed'].astype(int)
sc_b2_gee['task_cat'] = sc_b2_gee['task'].astype(str)
sc_b2_gee['trial_c'] = sc_b2_gee['trial_n'] - sc_b2_gee['trial_n'].mean()

print(pd.crosstab(sc_b2_gee['task_cat'], sc_b2_gee['pressed']))
print()
print()

gee_model = smf.gee(
    formula='pressed ~ trial_c * task_cat',
    groups='subject_id',
    data=sc_b2_gee,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)
gee_res = gee_model.fit()

print('H2 Secondary Test — Logistic GEE Summary : pressed ~ trial_c * task_cat')
print(gee_res.summary())
print()
print()
coef = gee_res.params

or_df = pd.DataFrame({
    'coef': gee_res.params,
    'OR': np.exp(gee_res.params),
    'CI_low': np.exp(gee_res.conf_int()[0]),
    'CI_high': np.exp(gee_res.conf_int()[1]),
    'p': gee_res.pvalues
})

print(or_df)

pressed      0     1
task_cat            
1         1450  1472
2         1671  1286


H2 Secondary Test — Logistic GEE Summary : pressed ~ trial_c * task_cat
                               GEE Regression Results                              
Dep. Variable:                     pressed   No. Observations:                 5879
Model:                                 GEE   No. clusters:                       70
Method:                        Generalized   Min. cluster size:                  69
                      Estimating Equations   Max. cluster size:                  97
Family:                           Binomial   Mean cluster size:                84.0
Dependence structure:         Exchangeable   Num. iterations:                     5
Date:                     Sat, 25 Apr 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         21:00:23
                            coef    std err          z      P>|z|      [0.025      0.9

Both the primary one-sample t-test, as well as the secondary GEE test show below-threshold significance, at both corrected and uncorrected levels. The test assumed that SC early-learning in Task 2 would be superior to Task 1. In fact, the direction of transfer is negative : 35.7% of subjects show positive transfer, while 50.0% show negative transfer (< 0). t(69) = -1.432, with Cohen's d = -0.171.

We also examined whether differences in learning rates (initially preregistered as a Linear Mixed Model, later reconsidered in favour of a logistic GEE) between Tasks could be examined. This was also negative, confirming H2 as null.

## H3: Observation priming

- Preregistration : Group A acquires SC contingency faster than Group B.  
- Statistic : independent samples t-test (Welch's) on SC acquisition rate (slope of pressed ~ exposure_n for SC trials), two-tailed.  
- Operationalisation : The preregistered 5-trial rolling window criterion could not be reliably computed due to R trial sparsity (~10% of block 2 trials). I took a rate-based alternative supported by the preregistration's Indices section: "Learning rate: trial-by-trial SC press rate within block 2, fit with [a] growth function per subject." Linear regression of press rate yields a slope capturing learning rate per exposure, invariant to starting point.  
- Rationale : Group A observed causal structure during block 1 (OC active, p = 0.90). Group B observed random outcomes (p = 0.10). If observing environmental structure primes contingency learning, Group A should show faster SC acquisition.

In [33]:
def compute_learning_slopes(subj_df, min_exposures=5):
    result = {'sc_slope': np.nan, 'r_slope': np.nan, 'sc_n': 0, 'r_n': 0}
    for role in ['SC', 'R']:
        role_data = subj_df[subj_df['trial_role'] == role].sort_values('trial_n').reset_index(drop=True)
        result[f'{role.lower()}_n'] = len(role_data)
        if len(role_data) >= min_exposures:
            result[f'{role.lower()}_slope'] = linregress(
                np.arange(1, len(role_data) + 1), role_data['pressed']
            ).slope
    return result

t1b2 = trials[(trials['task'] == 1) & (trials['block'] == 2)].copy()
slope_df = pd.DataFrame([
    {**compute_learning_slopes(t1b2[t1b2['subject_id'] == s]), 'subject_id': s}
    for s in t1b2['subject_id'].unique()
]).merge(subject_meta[['subject_id', 'group', 'sc_prop']], on='subject_id')

valid_slopes = slope_df[slope_df['sc_slope'].notna() & slope_df['r_slope'].notna()].copy()


# Trial distribution per property in T1 B2

print("Trial distribution (Task 1 Block 2):")
print(f"SC: M = {slope_df['sc_n'].mean():.1f}, range [{slope_df['sc_n'].min()}, {slope_df['sc_n'].max()}]")
print(f"R:  M = {slope_df['r_n'].mean():.1f}, range [{slope_df['r_n'].min()}, {slope_df['r_n'].max()}]")
print(f"Subjects excluded (< 5 trials in either condition): {len(slope_df) - len(valid_slopes)}")
print()


# Primary H3 question : SC slope, Group A vs B

grp_a_sc = valid_slopes[valid_slopes['group'] == 'A']['sc_slope']
grp_b_sc = valid_slopes[valid_slopes['group'] == 'B']['sc_slope']
t_h3, p_h3 = ttest_ind(grp_a_sc, grp_b_sc, equal_var=False)
d_h3 = cohens_d_ind(grp_a_sc, grp_b_sc)

print("SC acquisition rate — Group A vs B (Task 1 Block 2)")
print(f"Group A (N={len(grp_a_sc)}): M = {grp_a_sc.mean():.5f}, SD = {grp_a_sc.std():.5f}")
print(f"Group B (N={len(grp_b_sc)}): M = {grp_b_sc.mean():.5f}, SD = {grp_b_sc.std():.5f}")
print()
print(f"t (Welch) = {t_h3:.3f}, p (two-tailed) = {p_h3:.4f}, Cohen's d = {d_h3:.3f}")
print(f"Significant at α = 0.05: {p_h3 < 0.05}")
print(f"Significant at Bonferroni α = 0.0125: {p_h3 < 0.0125}")
print()
print()

# Exploratory : Learning direction (collapsed across groups A and B)

print("Exploratory - learning direction during Block 2 (collapsed across groups A and B)")

sc_slopes = valid_slopes['sc_slope']
r_slopes = valid_slopes['r_slope']
t_sc, p_sc = ttest_1samp(sc_slopes, 0)
t_r, p_r = ttest_1samp(r_slopes, 0)

print(f"SC slope: M = {sc_slopes.mean():.4f}, t = {t_sc:.3f}, p = {p_sc/2:.4f} (one-tail > 0), d = {sc_slopes.mean()/sc_slopes.std():.3f}")
print(f"R slope:  M = {r_slopes.mean():.4f}, t = {t_r:.3f}, p = {p_r/2:.4f} (one-tail < 0), d = {r_slopes.mean()/r_slopes.std():.3f}")
print()
print()

# Exploratory : learning magnitude comparison

abs_sc = valid_slopes['sc_slope'].abs()
abs_r = valid_slopes['r_slope'].abs()
diff = abs_r - abs_sc
t_mag, p_mag = ttest_rel(abs_r, abs_sc)
d_mag = diff.mean() / diff.std(ddof=1)

print("Exploratory - |R slope| vs |SC slope| (paired comparison)")
print(f"|SC slope|: M = {abs_sc.mean():.4f}, SD = {abs_sc.std():.4f}")
print(f"|R slope|:  M = {abs_r.mean():.4f}, SD = {abs_r.std():.4f}")
print(f"Diff (|R| - |SC|): M = {diff.mean():.4f}, SD = {diff.std():.4f}")
print()
print(f"t({len(diff)-1}) = {t_mag:.3f}, p = {p_mag/2:.4f} (one-tail |R| > |SC|), d = {d_mag:.3f}")

Trial distribution (Task 1 Block 2):
SC: M = 41.7, range [33, 48]
R:  M = 9.7, range [3, 17]
Subjects excluded (< 5 trials in either condition): 1

SC acquisition rate — Group A vs B (Task 1 Block 2)
Group A (N=38): M = 0.00305, SD = 0.01002
Group B (N=31): M = 0.00237, SD = 0.01093

t (Welch) = 0.268, p (two-tailed) = 0.7899, Cohen's d = 0.065
Significant at α = 0.05: False
Significant at Bonferroni α = 0.0125: False


Exploratory - learning direction during Block 2 (collapsed across groups A and B)
SC slope: M = 0.0027, t = 2.200, p = 0.0156 (one-tail > 0), d = 0.265
R slope:  M = -0.0293, t = -3.459, p = 0.0005 (one-tail < 0), d = -0.416


Exploratory - |R slope| vs |SC slope| (paired comparison)
|SC slope|: M = 0.0080, SD = 0.0071
|R slope|:  M = 0.0534, SD = 0.0542
Diff (|R| - |SC|): M = 0.0454, SD = 0.0541

t(68) = 6.969, p = 0.0000 (one-tail |R| > |SC|), d = 0.839


The primary H3 test found no evidence that observing causal structure in block 1 accelerates SC acquisition in block 2 (t = 0.27, p = 0.79, Cohen's d = 0.065).
 
Further exploratory analysis of learning dynamics (collapsed across groups A and B) revealed an asymmetry: discrimination during block 2 emerges primarily through R suppression (slope = -0.029, d = -0.42, p < 0.001) rather than SC acquisition (slope = 0.003, d = 0.27, p = 0.016). 
A paired comparison of slope magnitudes confirmed this as a large effect (t = 6.97, p = 0.0000, Cohen's d = 0.84)

This challenges the theoretical premise of the initial hypothesis : while subjects do learn to respond to SC, the dominant learning signal and driver of change seems to come from suppression of responses to random-outcome stimuli. Under this interpretation, discrimination would emerge through avoidance more than approach.

## H4: Belief persistence 

- Prediction : block 3 SC press rate > block 1 baseline press rate*
- Statistic : paired t-test (per pre-registration), one-tailed (block 3 > block 1).  
- Rationale : if contingency is learnt, the belief would persist without direct outcome observations.

***NOTE** : block 1 is watch-only (spacebar disabled) - the wrong comparison block was entered during the preregistration. It was meant to imply the 'baseline' of SC press rates rather than the 'watch only' training block. The `pressed` column for all block 1 trials is 0 by design — confirmed in data (all 1,400 block-1 rows have `pressed = 0`).   

To compute the difference in press rates, the mean SC press rate values across block 2 and end-of-block-2 press rates (10 last trials) were taken, and compared to the average block 3 press rate. This perovided a thorough baseline for comparison. 

Block 3 press rates are averaged across tasks 1 and 2 per subject (consistent with H1).

In [38]:
def get_b2_stats(subj_trials, n_late=10):
    role_data = subj_trials[subj_trials['trial_role'] == 'SC'].sort_values('trial_n')
    if len(role_data) >= n_late:
        return {
            'sc_b2_all': role_data['pressed'].mean(),
            'sc_b2_late': role_data.tail(n_late)['pressed'].mean()
        }
    return {'sc_b2_all': np.nan, 'sc_b2_late': np.nan}

def get_b3_slope(subj_trials, min_trials=5):
    role_data = subj_trials[subj_trials['trial_role'] == 'SC'].sort_values('trial_n').reset_index(drop=True)
    if len(role_data) >= min_trials:
        return linregress(np.arange(1, len(role_data) + 1), role_data['pressed']).slope
    return np.nan

b2_trials = trials[trials['block'] == 2].copy()
b3_trials = trials[trials['block'] == 3].copy()
persist_df = pd.DataFrame({'subject_id': trials['subject_id'].unique()})
b2_stats = persist_df['subject_id'].apply(
    lambda s: pd.Series(get_b2_stats(b2_trials[b2_trials['subject_id'] == s]))
)

persist_df['sc_b2_all'] = b2_stats['sc_b2_all'].values
persist_df['sc_b2_late'] = b2_stats['sc_b2_late'].values
persist_df['sc_b3'] = (
    b3_trials[b3_trials['trial_role'] == 'SC']
    .groupby('subject_id')['pressed'].mean()
    .reindex(persist_df['subject_id']).values
)

persist_df['sc_b3_slope'] = persist_df['subject_id'].apply(
    lambda s: get_b3_slope(b3_trials[b3_trials['subject_id'] == s])
)

persist_df['sc_change_all'] = persist_df['sc_b3'] - persist_df['sc_b2_all']
persist_df['sc_change_late'] = persist_df['sc_b3'] - persist_df['sc_b2_late']

persist_df = persist_df.merge(subject_meta[['subject_id', 'group']], on='subject_id')
valid = persist_df.dropna(subset=['sc_b2_all', 'sc_b2_late', 'sc_b3'])


# Primary Test - SC persistence (B2 to B3)

print("H4 Primary: SC press rate persistence (B2 to B3)")
print(f"N = {len(valid)}")

t_h4_all, p_h4_all = ttest_rel(valid['sc_b3'], valid['sc_b2_all'])
d_h4_all = valid['sc_change_all'].mean() / valid['sc_change_all'].std(ddof=1)

print(f"B2 all: M = {valid['sc_b2_all'].mean():.3f}, SD = {valid['sc_b2_all'].std():.3f}")
print(f"B3: M = {valid['sc_b3'].mean():.3f}, SD = {valid['sc_b3'].std():.3f}")
print(f"Change: M = {valid['sc_change_all'].mean():+.3f}, SD = {valid['sc_change_all'].std():.3f}")
print()
print(f"t({len(valid)-1}) = {t_h4_all:.3f}, p = {p_h4_all:.4f}, d = {d_h4_all:.3f}")
print(f"Significant at α = {alpha}: {p_h4_all < alpha}")
print(f"Significant at α = {alpha_bonferroni}: {p_h4_all < alpha_bonferroni}")
print()
print()


# Secondary Test - SC persistence (B2 late trialsto B3)

print("H4 Secondary Test - SC press rate persistence (B2 late trials to B3)")

t_h4_late, p_h4_late = ttest_rel(valid['sc_b3'], valid['sc_b2_late'])
d_h4_late = valid['sc_change_late'].mean() / valid['sc_change_late'].std(ddof=1)

print(f"B2 (last 10 trials) : M = {valid['sc_b2_late'].mean():.3f}, SD = {valid['sc_b2_late'].std():.3f}")
print(f"B3 : M = {valid['sc_b3'].mean():.3f}, SD = {valid['sc_b3'].std():.3f}")
print(f"Change : M = {valid['sc_change_late'].mean():+.3f}, SD = {valid['sc_change_late'].std():.3f}")
print()
print(f"t({len(valid)-1}) = {t_h4_late:.3f}, p = {p_h4_late:.4f}, d = {d_h4_late:.3f}")
print(f"Significant at α = {alpha}: {p_h4_late < alpha}")
print(f"Significant at α = {alpha_bonferroni}: {p_h4_late < alpha_bonferroni}")
print()
print()

# Within-B3 slope

print("Exploratory Test - within-B3 slope (SC only)")

slopes = persist_df['sc_b3_slope'].dropna()
t_slope, p_slope = ttest_1samp(slopes, 0)
d_slope = slopes.mean() / slopes.std(ddof=1)

print(f"N = {len(slopes)}")
print(f"SC slope: M = {slopes.mean():+.4f}, SD = {slopes.std():.4f}")
print()
print(f"t({len(slopes)-1}) = {t_slope:.3f}, p = {p_slope:.4f}, d = {d_slope:.3f}")
print(f"Significant at α = {alpha}: {p_slope < alpha}")
print()
print()

# Exploratory Test - by Group

print("Exploratory Test - SC persistence by Group")
print()
for group in ['A', 'B']:
    g = valid[valid['group'] == group]
    g_slopes = persist_df[(persist_df['group'] == group) & persist_df['sc_b3_slope'].notna()]['sc_b3_slope']
    print(f"Group {group} (N={len(g)}):")
    print(f"B2 all: M = {g['sc_b2_all'].mean():.3f}, SD = {g['sc_b2_all'].std():.3f}")
    print(f"B2 late: M = {g['sc_b2_late'].mean():.3f}, SD = {g['sc_b2_late'].std():.3f}")
    print(f"B3: M = {g['sc_b3'].mean():.3f}, SD = {g['sc_b3'].std():.3f}")
    print(f"Change (all): M = {g['sc_change_all'].mean():+.3f}, SD = {g['sc_change_all'].std():.3f}")
    print(f"Change (late): M = {g['sc_change_late'].mean():+.3f}, SD = {g['sc_change_late'].std():.3f}")
    print(f"B3 slope (N={len(g_slopes)}): M = {g_slopes.mean():+.4f}, SD = {g_slopes.std():.4f}")
    print()

H4 Primary: SC press rate persistence (B2 to B3)
N = 70
B2 all: M = 0.466, SD = 0.230
B3: M = 0.515, SD = 0.252
Change: M = +0.049, SD = 0.147

t(69) = 2.785, p = 0.0069, d = 0.333
Significant at α = 0.05: True
Significant at α = 0.0125: True


H4 Secondary Test - SC press rate persistence (B2 late trials to B3)
B2 (last 10 trials) : M = 0.499, SD = 0.293
B3 : M = 0.515, SD = 0.252
Change : M = +0.016, SD = 0.192

t(69) = 0.714, p = 0.4774, d = 0.085
Significant at α = 0.05: False
Significant at α = 0.0125: False


Exploratory Test - within-B3 slope (SC only)
N = 70
SC slope: M = -0.0008, SD = 0.0085

t(69) = -0.817, p = 0.4165, d = -0.098
Significant at α = 0.05: False


Exploratory Test - SC persistence by Group

Group A (N=39):
B2 all: M = 0.432, SD = 0.235
B2 late: M = 0.462, SD = 0.284
B3: M = 0.494, SD = 0.245
Change (all): M = +0.063, SD = 0.151
Change (late): M = +0.033, SD = 0.181
B3 slope (N=39): M = -0.0007, SD = 0.0084

Group B (N=31):
B2 all: M = 0.509, SD = 0.220
B2 late:

H4 is supported, with nuance. When comparing block 3 SC press rate to the full block 2 mean, there is a significant increase (B2: 0.466 → B3: 0.515; t = 2.79, p = 0.007, d = 0.33). However, when comparing to the last 10 trials of block 2 (the state at the end of active learning), there is no significant change (B2 late: 0.499 → B3: 0.515; t = 0.71, p = 0.48, d = 0.09).

This pattern indicates that average SC beliefs formed throughout block 2 *persist without decay* when feedback is removed. The increase relative to B2 reflects the learning trajectory within block 2 (SC slope was positive), not continued learning in block 3. Within-block-3 slopes were flat (M = −0.0008, p = 0.42), confirming stability rather than diffusion or extinction.

Both groups showed similar patterns: Group A changed +0.033 from B2 late, Group B changed −0.004. The absence of decay in either group suggests that contingency beliefs, once acquired, are stable over short timescales without reinforcement. Whether this is due to active maintenance of the belief versus a freeze without further evidence cannot be confidently stated.

---
## Results summary

In [39]:
summary = pd.DataFrame([
    {
        'Hypothesis':       'H1: SC > R (block 2)',
        'Test':             'Paired t',
        'N':                70,
        't':                2.239,
        'df':               69,
        'p':                0.0142,
        'p type':           'one-tailed',
        "d (Cohen's)":      0.268,
        'sig α=0.05':       True,
        'sig α=0.0125':     False,
    },
    {
        'Hypothesis':       'H2: transfer index > 0',
        'Test':             'One-sample t',
        'N':                70,
        't':                -1.432,
        'df':               69,
        'p':                0.9217,
        'p type':           'one-tailed',
        "d (Cohen's)":      -0.171,
        'sig α=0.05':       False,
        'sig α=0.0125':     False,
    },
    {
        'Hypothesis':       'H3: Group A vs B (SC slope)',
        'Test':             'Indep t (Welch)',
        'N':                '38A/31B',
        't':                0.268,
        'df':               67.0,
        'p':                0.7899,
        'p type':           'two-tailed',
        "d (Cohen's)":      0.065,
        'sig α=0.05':       False,
        'sig α=0.0125':     False,
    },
    {
        'Hypothesis':       'H4: SC B2 all → B3',
        'Test':             'Paired t',
        'N':                70,
        't':                2.785,
        'df':               69,
        'p':                0.0069,
        'p type':           'two-tailed',
        "d (Cohen's)":      0.333,
        'sig α=0.05':       True,
        'sig α=0.0125':     True,
    },
    {
        'Hypothesis':       'H4: SC B2 late → B3',
        'Test':             'Paired t',
        'N':                70,
        't':                0.714,
        'df':               69,
        'p':                0.4774,
        'p type':           'two-tailed',
        "d (Cohen's)":      0.085,
        'sig α=0.05':       False,
        'sig α=0.0125':     False,
    },
])

print('RESULTS SUMMARY — Pre-registered tests H1–H4')
print()
print(summary.to_string(index=False))
print()
print('Conclusions:')
print('H1: Significant at α=0.05 but not after Bonferroni correction.')
print('H2: Negative transfer index — opposite to predicted direction.')
print('H3: Preregistered TTC measure not computable (R sparsity); SC slope used instead.')
print('H4: Two comparisons reported:')
print(' - B2 all → B3: significant increase (d=0.33), reflects learning within B2.')
print(' - B2 late → B3: no change (d=0.09), confirms persistence of learned state.')
print('Group × SC property imbalance noted (A: 18 velocity-SC, B: 5).')

RESULTS SUMMARY — Pre-registered tests H1–H4

                 Hypothesis            Test       N      t   df      p     p type  d (Cohen's)  sig α=0.05  sig α=0.0125
       H1: SC > R (block 2)        Paired t      70  2.239 69.0 0.0142 one-tailed        0.268        True         False
     H2: transfer index > 0    One-sample t      70 -1.432 69.0 0.9217 one-tailed       -0.171       False         False
H3: Group A vs B (SC slope) Indep t (Welch) 38A/31B  0.268 67.0 0.7899 two-tailed        0.065       False         False
         H4: SC B2 all → B3        Paired t      70  2.785 69.0 0.0069 two-tailed        0.333        True          True
        H4: SC B2 late → B3        Paired t      70  0.714 69.0 0.4774 two-tailed        0.085       False         False

Conclusions:
H1: Significant at α=0.05 but not after Bonferroni correction.
H2: Negative transfer index — opposite to predicted direction.
H3: Preregistered TTC measure not computable (R sparsity); SC slope used instead.
H4: Tw